# P06 · tendon vibration × polarity, 29-09-2026

**Why this session has its own notebook.** P06 is the **only participant whose vibration session
was run in both polarities**. P03 and P04 were cathodic only, so their vibration blocks sit in
`P03_P04_P06_tendon_vibration.ipynb`, where P06's cathodic recordings join them for a
three-participant comparison. Here the question is different and it can only be asked of P06:

> **Does tendon vibration act the same way on cathodic and on anodic stimulation?**

If the two polarities recruit different fibres — which the polarity notebooks suggest they might —
then a manipulation that works through **Ia afferents** should not touch them equally. Tendon
vibration is exactly that manipulation: it produces presynaptic inhibition of Ia terminals, so a
Ia-mediated response should be suppressed while a directly evoked one should not.

**The design.** 27 recordings: 3 blocks × 3 polarities × 3 protocols.

| | |
|---|---|
| **blocks** | Baseline → VIB off (vibrator strapped on, not running) → VIB ON |
| **polarities** | cathodic (*negative* in the log), anodic (*positive*), biphasic |
| **protocols** | 1 Hz burst, 30 Hz burst, ARC-EX |
| **vibrator** | **left wrist extensor** — the same site as P03 |

**VIB off is the control that matters.** The vibrator is attached but silent, so whatever it shows
is drift between blocks — time, electrode contact, the participant settling. A VIB ON effect has
to beat that, not just beat Baseline.

**What to look at.** `Ext. digitorum (L)` is the vibrated muscle; the other three are the internal
control. A real effect is VIB ON differing from Baseline **at the vibrated muscle and not at the
others**, by more than VIB off drifts.

**The 1 Hz blocks are the extra prize here.** Nobody else has them. At 1 s between pulses there is
no post-activation depression left, so the 1 Hz response is the cleanest single-pulse measure in
the whole dataset — and comparing it against 30 Hz in the same block is a genuine frequency
comparison, which the 30 Hz vs ARC-EX one is not (those are both 30 Hz).

**Colours:** 30 Hz **grey**, ARC-EX **red**, 1 Hz **blue**; plain = cathodic, hatched = anodic.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from functions import (set_style, load_run, pretty, thresholds_for, threshold_source,
                       condition_table, fig_vibration, fig_bars,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       mt_stamp, warn_if_stale,
                       fig_threshold_grid, fig_detection_grid,
                       fig_group_summary, fig_group_panels, snr_verdict, fig_noise_check)
set_style()
from functions.paper import motor_thresholds

## 1 · Config

In [ ]:
# ======== the recordings ========
# P06, 29-09-2026. That folder ALSO holds a copy of the whole 28-09 session, so only the files
# stamped 20260929 belong here - every one below is named explicitly.
D = "tSCS_CHUV_data/29-09-2026/"

ALL = {   # (protocol, polarity, block) -> csv.  "negative" in the log = cathodic, "positive" = anodic
 ("1 Hz",   "cathodic", "Baseline"): D + "Burst_1Hz_autosave_20260929_154415_417ms.csv",       # 15:44
 ("1 Hz",   "cathodic", "VIB off"):  D + "Burst_1Hz_autosave_20260929_162950_616ms.csv",       # 16:29
 ("1 Hz",   "cathodic", "VIB ON"):   D + "Burst_1Hz_autosave_20260929_180111_780ms.csv",       # 18:01
 ("1 Hz",   "anodic",   "Baseline"): D + "Burst_1Hz_autosave_20260929_154124_545ms.csv",       # 15:41
 ("1 Hz",   "anodic",   "VIB off"):  D + "Burst_1Hz_autosave_20260929_162709_372ms.csv",       # 16:27
 ("1 Hz",   "anodic",   "VIB ON"):   D + "Burst_1Hz_autosave_20260929_165059_014ms.csv",       # 16:50
 ("1 Hz",   "biphasic", "Baseline"): D + "Burst_1Hz_Biphasic_autosave_20260929_154957_624ms.csv",
 ("1 Hz",   "biphasic", "VIB off"):  D + "Burst_1Hz_Biphasic_autosave_20260929_163227_227ms.csv",
 ("1 Hz",   "biphasic", "VIB ON"):   D + "Burst_1Hz_Biphasic_autosave_20260929_172408_725ms.csv",
 ("30 Hz",  "cathodic", "Baseline"): D + "Burst_autosave_20260929_155626_234ms.csv",           # 15:56
 ("30 Hz",  "cathodic", "VIB off"):  D + "Burst_autosave_20260929_163809_190ms.csv",           # 16:38
 ("30 Hz",  "cathodic", "VIB ON"):   D + "Burst_autosave_20260929_175220_421ms.csv",           # 17:52
 ("30 Hz",  "anodic",   "Baseline"): D + "Burst_autosave_20260929_155345_812ms.csv",           # 15:53
 ("30 Hz",  "anodic",   "VIB off"):  D + "Burst_autosave_20260929_163517_916ms.csv",           # 16:35
 ("30 Hz",  "anodic",   "VIB ON"):   D + "Burst_autosave_20260929_170919_038ms.csv",           # 17:09
 ("30 Hz",  "biphasic", "Baseline"): D + "Burst_Biphasic_autosave_20260929_155921_927ms.csv",
 ("30 Hz",  "biphasic", "VIB off"):  D + "Burst_Biphasic_autosave_20260929_164040_213ms.csv",
 ("30 Hz",  "biphasic", "VIB ON"):   D + "Burst_Biphasic_autosave_20260929_171716_204ms.csv",
 ("ARC-EX", "cathodic", "Baseline"): D + "Modulated_autosave_20260929_160545_334ms.csv",       # 16:05
 ("ARC-EX", "cathodic", "VIB off"):  D + "Modulated_autosave_20260929_164440_099ms.csv",       # 16:44
 ("ARC-EX", "cathodic", "VIB ON"):   D + "Modulated_autosave_20260929_170027_460ms.csv",       # 17:00
 ("ARC-EX", "anodic",   "Baseline"): D + "Modulated_autosave_20260929_160150_049ms.csv",       # 16:01
 ("ARC-EX", "anodic",   "VIB off"):  D + "Modulated_autosave_20260929_164216_867ms.csv",       # 16:42
 ("ARC-EX", "anodic",   "VIB ON"):   D + "Modulated_autosave_20260929_173301_845ms.csv",       # 17:33
 ("ARC-EX", "biphasic", "Baseline"): D + "Modulated_Biphasic_autosave_20260929_160823_040ms.csv",
 ("ARC-EX", "biphasic", "VIB off"):  D + "Modulated_Biphasic_autosave_20260929_164704_430ms.csv",
 ("ARC-EX", "biphasic", "VIB ON"):   D + "Modulated_Biphasic_autosave_20260929_174216_620ms.csv",
}

# ======== what to analyse ========
PROTOCOLS  = ["30 Hz", "ARC-EX"]              # add "1 Hz" for the frequency comparison
POLARITIES = ["cathodic", "anodic"]           # add "biphasic" for all three
# =================================

BLOCKS     = ["Baseline", "VIB off", "VIB ON"]          # everything that was recorded
CONTROL    = "VIB off"                                  # what the analysis compares against
CONDITIONS = [CONTROL, "VIB ON"]                        # the bars inside every panel
# The threshold of each group comes from its CONTROL recording, and that group's blocks are read
# at it, muscle by muscle.
#
# VIB off is the control because it is the only one that isolates the vibration. The vibrator is
# strapped on and silent, so it shares the setup, the electrode positions and the elapsed time
# with VIB ON and differs in one thing: whether the motor is running. Baseline sits 30-60 min
# earlier with nothing attached, and the gap between Baseline and VIB off turned out to be larger
# than anything VIB ON adds - some muscles were already at 6 % of Baseline before the vibrator was
# switched on. Measured against Baseline the figure is mostly reporting that drift.
#
# Baseline stays in BLOCKS so sections 2 and 3 still show it. Set CONTROL = "Baseline" and
# CONDITIONS = BLOCKS to put it back in the results.
PANELS     = [f"{p} · {pol}" for p in PROTOCOLS for pol in POLARITIES]   # one panel each
RUN        = {(f"{p} · {pol}", b): ALL[(p, pol, b)]
              for p in PROTOCOLS for pol in POLARITIES for b in BLOCKS}

VIBRATED = "Ext. digitorum (L)"      # the vibrator sat on the left wrist extensor
SITE     = "left wrist extensor"

# the vibrator is the thing that changes, so it gets the colour: VIB off grey, VIB ON orange.
# Baseline stays pale grey for sections 2 and 3, where it is still shown.
COND_COLOUR = {"Baseline": "#D4D4D4", "VIB off": "#8C8C8C", "VIB ON": "#E67E22"}
LINES       = {"Baseline": (0, (1, 1)), "VIB off": "-", "VIB ON": "-"}   # the EMG traces

MUSCLES = ["Flex. digitorum (L)", "Flex. digitorum (R)", "Ext. digitorum (L)",
           "Flex. carpi rad. (R)"]

# ======== the signal-to-noise gate ========
SNR_GATE = None     # None = draw every reading. 2.0 = leave out anything under twice its noise.
# The gate divides by the PRE-STIMULUS noise, and in this session that noise is partly the
# vibrator: it is a motor on the skin over Ext. digitorum (L) and it raises that electrode's
# noise floor 2-8x (section 3). The noise check before section 4c shows what the gate was
# throwing away - ten superimposed pulses with a clear peak and trough, flagged at 1.5x only
# because the denominator had grown. So it is off, and the readings are judged by eye instead.
# Set it to 2.0 to see which ones it would have removed.
SNR_SHOW = 2.0      # still LISTS anything under this in the noise check, for inspection
# ==========================================

KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)

runs = RUN
for pan in PANELS:
    for b in BLOCKS:
        f = runs[(pan, b)]
        amps = [m["amp_ma"] for m in load_run(f)[0]]
        print(f"{pan:20s} {b:10s} {f.split('/')[-1][-22:]}  {amps[0]:g}-{amps[-1]:g} mA, "
              f"step {amps[1] - amps[0]:g}" + ("   <- control" if b == CONTROL else ""))

## 2 · The motor thresholds you chose

**Only the two blocks the results use are shown** — `VIB off` and `VIB ON`. Baseline is not
analysed, so it is not here; it is still in the config and in the noise table if you want it.

Each group takes the threshold picked in **its own `VIB off` recording**, and both of its blocks
are read at that one intensity, muscle by muscle.

Two grids:

1. **what is saved for each recording** — the first one, read straight from
   `results/P06_2026-09-29_tendon_vibration/mt_*.csv`, so nothing can quietly substitute a value;
2. **what the analysis uses** — the second, below the numbers, with the line on that group's
   `VIB off` threshold in **both** rows. That is the intensity section 4 measures at, so the
   `VIB ON` row shows the sweep with the control's threshold marked on it, not its own.

Where the two differ, the second is the one the results came from.

In [ ]:
# only the blocks the results use, and the line drawn in each is the intensity the analysis
# reads it at - that group's VIB off threshold, in the VIB ON row as well as the VIB off one
CHECK = {f"{pan} · {b}": runs[(pan, b)] for pan in PANELS for b in CONDITIONS}

picked = fig_threshold_grid(CHECK, MUSCLES,
                            title="P06 · 29-09 — the VIB off threshold used in each group")

### The numbers, and where each one came from

In [ ]:
MT = {}      # MT[panel] = {muscle: mA}, from that panel's own CONTROL recording (VIB off)
for pan in PANELS:
    csv = runs[(pan, CONTROL)]
    MT[pan] = thresholds_for(csv, MUSCLES, consecutive=2, **KW)
    print(f"{pan:20s} {CONTROL:10s} {threshold_source(csv)}")

MT_ALL = {lab: picked.get(lab, {}) for lab in CHECK}

# redraw the grid with the line on the intensity each block is MEASURED at, not on its own pick
USED = {f"{pan} · {b}": MT[pan] for pan in PANELS for b in CONDITIONS}
_ = fig_threshold_grid(CHECK, MUSCLES, thresholds=USED,
                       title="the same recordings, with the line on the intensity each is read at")

print("\nmotor threshold (mA), as picked in each recording")
print(f"{'recording':26s}" + "".join(f"{m.split(' (')[0][:9] + m[-3:]:>16s}" for m in MUSCLES))
for lab in CHECK:
    print(f"{lab:26s}" + "".join(f"{str(MT_ALL[lab].get(m) or '-'):>16s}" for m in MUSCLES))

MT_STAMP = mt_stamp(runs)

## 3 · Are the peaks detected correctly? — every recording, every muscle

The same grid as section 2, showing how the peaks are found **at the intensity section 4
measures** — that is, every block of a group at that group's Baseline threshold. Switch
`AT_ANALYSIS_INTENSITY` to `False` to see each recording at its own pick instead.

The 10 pulses re-aligned on their own onset: the 10 pulses re-aligned on their own onset, with the max (v) and min (^) the
peak-to-peak is made of.

| | |
|---|---|
| **green** | the response window |
| **red bands** | the only places a peak may be taken, `ANCHOR_WIN_MS` either side of the train's own average latency |
| red ring | flagged — on a window border, or at a different latency |
| corner | `clean`, or how many of the 10 pulses were flagged |

**At 1 Hz there is only one pulse per second**, so a 1 Hz recording has a single response per
train and the train-shape measures do not apply to it — check it here, read it in section 4 as a
1st-pulse amplitude only.

`TRY` loosens the settings **for this figure only**; copy anything you keep into `KW` in
section 1.

In [ ]:
# ======== which intensity to inspect ========
AT_ANALYSIS_INTENSITY = True   # True  = every block at its group's CONTROL threshold, which is
                               #         what section 4 actually measures
                               # False = every recording at its own pick (needs the VIB rows
                               #         picked, and shows something the analysis does not use)
TRY = dict(KW)
# TRY["anchor_win_ms"] = 6.0
# TRY["anchor"] = None
# TRY["resp_start_ms"] = 6.0
# TRY["min_snr"] = None
# ============================================

DET_MT = ({f"{pan} · {b}": MT[pan] for pan in PANELS for b in BLOCKS}
          if AT_ANALYSIS_INTENSITY else MT_ALL)

fig_detection_grid(CHECK, DET_MT, MUSCLES, edge_ms=1.0, jitter_ms=0.5,
                   title=("P06 · 29-09 — peak detection at each group's VIB off threshold"
                          if AT_ANALYSIS_INTENSITY else
                          "P06 · 29-09 — peak detection at each recording's own threshold"),
                   **TRY)

### The vibrator's own noise — check this before reading section 4

The vibrator is a motor strapped to the skin over **Ext. digitorum (L)**, and its mechanical
noise lands in that electrode. The table below is the pre-stimulus noise floor of each muscle in
each block, in mV, with its change from the control block.

**This is the reason the vibrated muscle keeps being labelled `no response`.** Its noise rises
while the muscles on the other arm stay put, so the ratio of response to noise falls even when
the response does not. Judge that muscle on its **millivolts** (section 4c) rather than on any
criterion that divides by noise.

In [ ]:
from functions.burst import noise_p2p as _noise, burst_p2p

print(f"{'panel':20s}{'block':10s}" + "".join(f"{m.split(' (')[0][:8] + m[-3:]:>17s}" for m in MUSCLES))
for pan in PANELS:
    ref = {}
    for b in BLOCKS:
        meta_, t_, sig_ = load_run(runs[(pan, b)])
        chans = [c for c in sig_ if c != "Trigger A"]
        res_ = burst_p2p(meta_, t_, sig_, chans, **{**KW, "min_snr": None, "max_edge_frac": None})
        nb_ = _noise(t_, sig_, chans, win_len_ms=res_["win"][0][1] - res_["win"][0][0])
        amps_ = list(res_["amps"])
        row = ""
        for m in MUSCLES:
            ch = next((c for c in chans if pretty(c) == m), None)
            th = MT[pan].get(m)
            if ch is None or not th or th not in amps_:
                row += f"{'-':>17s}"; continue
            v = np.asarray(nb_[ch], float)
            v = float(v[amps_.index(th)]) if v.ndim else float(v)
            ref.setdefault(m, v)
            mark = "*" if m == VIBRATED else " "
            row += f"{v:9.4f} x{v / ref[m]:4.1f}{mark}"
        print(f"{pan:20s}{b:10s}{row}")
print("\n* = the vibrated muscle.  x = the noise relative to that group's control block.")

## 4 · The results

### 4a · The vibration effect, one panel per protocol and polarity

**VIB ON as a % of `VIB off`**, both read at the `VIB off` threshold of that group, muscle by
muscle. `*` marks the vibrated muscle. **100 % = the vibration changed nothing.**

`VIB off` is the control because it isolates the vibration: the vibrator is strapped on and
silent, so it shares the setup, the electrode positions and the elapsed time with `VIB ON` and
differs only in whether the motor runs. Baseline is 30–60 min earlier with nothing attached, and
the step from Baseline to VIB off is larger than anything VIB ON adds — some muscles were already
at 6 % of Baseline before the vibrator was switched on. Baseline is still in sections 2 and 3.

#### What a missing bar does and does not mean

A bar is left out when its response is under **2× the noise of that recording**, and the axis says
which: `< 2x noise`, or `under its noise` when it is smaller than the noise itself.

**Neither says there was no response.** The noise is not a constant here: the vibrator is a motor
on the skin over **Ext. digitorum (L)** and it raises that electrode's noise floor **2–8×** (the
table above). A train can fall under its own noise with its response unchanged — the vibrated
muscle's first pulse went **0.073 → 0.043 → 0.095 mV** across the three blocks while its ratio to
noise fell **5.5 → 1.6 → 1.3**. The response grew; only the ratio collapsed.

- **The three unvibrated muscles** keep a steady noise floor (within ±20 %), so a missing bar
  there does mean the response was too small to measure.
- **Ext. digitorum (L)** mostly goes missing because the vibrator was running. Judge it on its
  **millivolts** in section 4c and its traces in section 3, never on a ratio.

**The comparison this notebook exists for** is cathodic against anodic within a protocol: if
vibration suppresses one polarity and not the other, the two are not recruiting the same thing.

In [ ]:
warn_if_stale(runs, MT_STAMP, used=[(pan, CONTROL) for pan in PANELS])

tab = condition_table(runs, MT, MUSCLES, CONDITIONS, base=CONTROL, match="common", **KW)
fig_vibration(tab, PANELS, MUSCLES, CONDITIONS, colours=COND_COLOUR,
              min_snr_ratio=SNR_GATE, zero_if_absent=True,   # a response that vanished is drawn at 0
              vibrated={pan: VIBRATED for pan in PANELS},
              ylabel=f"% of {CONTROL}", ylim=(0, 200),
              titles={pan: f"{pan}   (vibrator on the {SITE})" for pan in PANELS},
              title=f"P06 · 29-09 — VIB ON as % of {CONTROL}, by protocol and polarity")

### The numbers, so they can be checked

In [ ]:
# the same verdict the figures used, so the table cannot disagree with them
print(f"{'panel':20s}{'muscle':22s}{'mA':>5s}" + "".join(f"{b:>16s}" for b in CONDITIONS))
for pan in PANELS:
    for m in MUSCLES:
        amp = next((tab[(pan, b, m)]["amp"] for b in CONDITIONS if (pan, b, m) in tab), None)
        if amp is None:
            continue
        cells = []
        for b in CONDITIONS:
            v, why = snr_verdict(tab, (pan, b, m), CONTROL, SNR_GATE)
            if why:
                cells.append(why)
            else:
                low = v.get("snr", 9e9) < SNR_SHOW      # drawn, but worth a look
                cells.append(f"{v['pct']:.0f} %" + (" ?" if low else "  "))
        star = " *" if m == VIBRATED else "  "
        print(f"{pan:20s}{m + star:22s}{amp:5g}" + "".join(f"{c_:>16s}" for c_ in cells))
print(f"\ncontrol = {CONTROL}; every block read at that group's {CONTROL} threshold.")
print(f"SNR_GATE = {SNR_GATE}  -> " + ("nothing is left out; every reading is drawn."
                                       if SNR_GATE is None else
                                       f"readings under {SNR_GATE:g}x their noise are left out."))
print(f"?  = under {SNR_SHOW:g}x its own noise - drawn, but check it in the noise panels above.")

### 4b · Each block against its own first pulse

Section 4a compares the **size** of the response between the two blocks. This asks a different
question of the same recordings: **inside each block, how much does the response fall over the
train?**

Every bar is a % of **that block's own 1st pulse** — `VIB off`'s 1st pulse is 100 % for the
`VIB off` bar, `VIB ON`'s own 1st pulse is 100 % for the `VIB ON` bar. Both blocks are still read
at the group's `VIB off` threshold, so the intensity is the same; only the normalisation changes.

**Why this one is more robust here.** A raised noise floor inflates the 1st pulse and the rest of
the train together, so it largely cancels in a within-train ratio, where it does not cancel at all
in a between-block one. That makes this the measure to trust for **Ext. digitorum (L)**, the
muscle under the vibrator — and the control block's own quality is irrelevant, so nothing is
dropped for `no control` here.

Two figures: the **mean of pulses 2–10**, and the **2nd pulse** alone.

In [ ]:
for key, lab in (("dep_rest", "mean of pulses 2-10"), ("dep2", "2nd pulse")):
    fig_vibration(tab, PANELS, MUSCLES, CONDITIONS, key=key,
                  colours=COND_COLOUR, min_snr_ratio=SNR_GATE, check_control=False,
                  vibrated={pan: VIBRATED for pan in PANELS},
                  ylabel="% of that block's own 1st pulse", ref=100, ylim=(0, 175),
                  titles={pan: pan for pan in PANELS},
                  title=f"P06 · 29-09 — {lab}, % of each block's own 1st pulse",
                  save=None)

print(f"{'panel':20s}{'muscle':22s}{'mA':>5s}"
      + "".join(f"{b + ' 2-10':>16s}{b + ' 2nd':>16s}" for b in CONDITIONS))
for pan in PANELS:
    for m in MUSCLES:
        amp = next((tab[(pan, b, m)]["amp"] for b in CONDITIONS if (pan, b, m) in tab), None)
        if amp is None:
            continue
        row = ""
        for b in CONDITIONS:
            v = tab.get((pan, b, m), {})
            for k in ("dep_rest", "dep2"):
                x = v.get(k, float("nan"))
                bad = SNR_GATE is not None and v.get("snr", float("nan")) < SNR_GATE
                row += f"{('--' if bad else (f'{x:.0f} %' if x == x else '-')):>16s}"
        star = " *" if m == VIBRATED else "  "
        print(f"{pan:20s}{m + star:22s}{amp:5g}{row}")
print("\n-- = that block's 1st pulse is under 2x its noise, so its own ratio is unreliable")

### Is it noise, or is there really nothing there? — decide by eye

A ratio cannot tell the two apart. A response of 1.3× its noise can be a small response buried in
a raised noise floor, or it can be nothing at all; both come out near 1. The only thing that
settles it is whether the **response window looks different from the silence before the stimulus**
in that same recording.

So this draws them side by side, on one scale:

| | |
|---|---|
| **left, red** | the 10 pulses, each over its response window |
| **right, grey** | windows of the **same length** taken from the 90 ms **before** the stimulus |

**If the two panels look alike, it is noise** — write that response down as absent. **If the left
has a deflection at a consistent latency that the right does not**, there is a response, however
small the ratio says it is, and the bar was dropped for the wrong reason.

The cell lists every reading section 4 left out and plots it. Set `ONLY` to inspect one.

In [ ]:
# ======== what to inspect ========
ONLY = None    # None = every reading section 4 dropped.  Or e.g.
               # ONLY = ("30 Hz · cathodic", "VIB ON", "Ext. digitorum (L)")
SNR  = SNR_SHOW
# =================================

flagged = [(pan, b, m) for pan in PANELS for b in CONDITIONS for m in MUSCLES
           if (pan, b, m) in tab and tab[(pan, b, m)].get("snr", 9e9) < SNR]
todo = [ONLY] if ONLY else flagged
print(f"{len(flagged)} reading(s) under {SNR:g}x noise"
      + ("  (the gate is OFF - all of them are in the figures)" if SNR_GATE is None
         else "  (the gate is ON - these are left out of the figures)")
      + (f"; showing 1" if ONLY else "") + "\n")
for pan, b, m in todo:
    v = tab.get((pan, b, m), {})
    fig_noise_check(runs[(pan, b)], m, v.get("amp"),
                    title=f"{pan} · {b} · {m}", **KW)

### 4c · Cathodic against anodic, the same effect side by side

Section 4a has one panel per protocol *and* polarity, which is four panels to compare by eye.
This re-slices the same numbers into the comparison the notebook is for: **one panel per protocol,
one bar per polarity**, so cathodic and anodic sit next to each other in the same muscle.

Two figures, **VIB off** and **VIB ON**, both as a % of their own panel's Baseline. Read them as a
pair: VIB off is the drift with the vibrator attached but silent, VIB ON is drift plus vibration.
**A vibration effect is VIB ON departing from Baseline by more than VIB off does, at the vibrated
muscle.**

There is one participant here, so these are single measurements — no error bars, nothing averaged.
The dots and SDs of the group figures elsewhere do not apply.

In [ ]:
# the same table, re-keyed so polarity is the bar and protocol is the panel
for block in ("VIB ON",):
    sliced = {(p, pol, m): tab[(f"{p} · {pol}", block, m)]
              for p in PROTOCOLS for pol in POLARITIES for m in MUSCLES
              if (f"{p} · {pol}", block, m) in tab}
    for key, lab in (("pct", "1st pulse"), ("pct_all", "all 10 pulses")):
        fig_vibration(sliced, PROTOCOLS, MUSCLES, POLARITIES, key=key,
                      min_snr_ratio=SNR_GATE, zero_if_absent=True,   # a bar under 2x its noise is left out and
                                           # the axis says whether it was weak or absent
                      colours={"cathodic": "#9A9A9A", "anodic": "#C0392B"},
                      hatches={"cathodic": "", "anodic": "///"},
                      vibrated={p: VIBRATED for p in PROTOCOLS},
                      ylabel=f"% of {CONTROL}", ylim=(0, 200),
                      titles={p: p for p in PROTOCOLS},
                      title=f"{block}, % of {CONTROL} — {lab}",
                      save=None)

### 4d · One figure per muscle, with the EMG underneath

The bars of section 4a muscle by muscle, with **the trains they were measured from drawn below
them**: `VIB off` in grey, `VIB ON` in orange, one panel per protocol and polarity, at that
group's `VIB off` threshold.

This is where the **vibrated muscle** has to be read. Its ratios are unusable — the vibrator
raises its noise floor several fold — but the millivolts and the traces are not. One participant,
one session, one electrode, so the **mV panel is directly comparable** between the bars here, which
it never is across participants.

Look at the traces for two things: whether the response is genuinely smaller under `VIB ON`, and
whether the trace between pulses is visibly noisier, which is the vibrator and not the cord.

In [ ]:
MT_cond = {(pan, c): {m: tab[(pan, c, m)]["amp"] for m in MUSCLES if (pan, c, m) in tab}
           for pan in PANELS for c in CONDITIONS}

for m in MUSCLES:
    fig_bars(runs, MT_cond, m, PANELS, protocols=tuple(CONDITIONS), steps=0,
             colours=COND_COLOUR, names={c: c for c in CONDITIONS},
             linestyles=LINES,
             min_snr_ratio=SNR_GATE, rest_from=2, with_p1=True, traces=True,
             title=f"{m}{'  *vibrated' if m == VIBRATED else ''}", save=None, **KW)

## 5 · Choose the motor thresholds

**Four cells, one per group** — each group's `VIB off` recording, which is the control everything
in section 4 is measured against. Move the slider, press **Set to slider**, then **Save**, and
re-run from section 2.

Nothing else needs picking: `Baseline` and `VIB ON` set no intensity. The threshold grid in
section 2 shows them if they happen to be saved, and section 3 inspects every block at its
group's `VIB off` threshold, which is what section 4 measures.

> To pick one anyway — say a `Baseline` you want on record — call `repick(("30 Hz · cathodic",
> "Baseline"))` in a new cell. It changes nothing downstream.

In [ ]:
def repick(key):
    """Open the picker for one recording, with a Save button.

    The green line is the DETECTOR's answer for this recording - not another block's, and not
    whatever is already saved (that is the orange trace)."""
    csv = runs[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
repick(('30 Hz · cathodic', 'VIB off'))

In [ ]:
repick(('30 Hz · anodic', 'VIB off'))

In [ ]:
repick(('ARC-EX · cathodic', 'VIB off'))

In [ ]:
repick(('ARC-EX · anodic', 'VIB off'))